[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/26_lora.ipynb)

# 🟠 Medium: LoRA (Low-Rank Adaptation)

Implement **LoRA** — parameter-efficient fine-tuning for large models.

$$h = W_0 x + \frac{\alpha}{r} B A x$$

### Signature
```python
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank, alpha=1.0): ...
    def forward(self, x: Tensor) -> Tensor: ...
```

### Requirements
- `self.linear`: frozen `nn.Linear` (weight & bias `requires_grad=False`)
- `self.lora_A`: `nn.Parameter(rank, in_features)` — random init
- `self.lora_B`: `nn.Parameter(out_features, rank)` — **zero** init
- Scaling: `alpha / rank`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.5 MB/s eta 0:00:00


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [7]:
# ✏️ YOUR IMPLEMENTATION HERE

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank, alpha=1.0):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features, bias=True)
        for p in self.linear.parameters():
          p.requires_grad = False
        self.lora_A = nn.Parameter(torch.zeros(rank, in_features), requires_grad=True)
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank), requires_grad=True)
        nn.init.kaiming_normal_(self.lora_A)
        self.scale = alpha / rank

    def forward(self, x):
        y = self.linear(x)
        # lora = F.linear(F.linear(x, self.lora_A), self.lora_B)
        lora = x @ self.lora_A.transpose(1, 0) @ self.lora_B.transpose(1, 0)
        return y + lora * self.scale

In [8]:
# 🧪 Debug
layer = LoRALinear(16, 8, rank=4)
x = torch.randn(2, 16)
print('Output:', layer(x).shape)
print('Trainable:', sum(p.numel() for p in layer.parameters() if p.requires_grad))
print('Total:    ', sum(p.numel() for p in layer.parameters()))

Output: torch.Size([2, 8])
Trainable: 96
Total:     232


In [9]:
# ✅ SUBMIT
from torch_judge import check
check('lora')


🧪 Testing: LoRA (Low-Rank Adaptation) (Medium)
──────────────────────────────────────────────────
  ✅ [1/5] Base weights frozen (2.0ms)
  ✅ [2/5] LoRA parameter shapes (0.5ms)
  ✅ [3/5] B=0 means output equals base (54.6ms)
  ✅ [4/5] Only LoRA params get gradients (25.4ms)
  ✅ [5/5] Forward computation (2.9ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (85.4ms total)
  Progress saved. Run status() to see your dashboard.

